In [1]:
using Pkg

Pkg.add("DataFrames")
Pkg.activate("Data_Assimilation")
Pkg.develop(path="../Krylov.jl")   # change path to local Krylov fork
Pkg.develop(path="../JSOSolvers.jl") 

   Resolving package versions...
    Updating `~/Desktop/DataAssim.jl/Project.toml`
  [a93c6f00] + DataFrames v1.8.1
    Updating `~/Desktop/DataAssim.jl/Manifest.toml`
  [a93c6f00] + DataFrames v1.8.1
  [842dd82b] + InlineStrings v1.4.5
  [41ab1584] + InvertedIndices v1.3.1
  [2dfb63ee] + PooledArrays v1.4.3
  [91c51154] + SentinelArrays v1.4.9
  [9fa8497b] + Future v1.11.0
Precompiling project...
   1200.2 ms  ✓ Measures
   1044.5 ms  ✓ StructUtils
   1043.4 ms  ✓ ManualNLPModels
   1046.2 ms  ✓ StableRNGs
    430.0 ms  ✓ Libmount_jll
    435.7 ms  ✓ Graphite2_jll
    493.1 ms  ✓ EpollShim_jll
    500.6 ms  ✓ LLVMOpenMP_jll
   1237.0 ms  ✓ JSOSolvers
    444.4 ms  ✓ Xorg_libICE_jll
    508.6 ms  ✓ libfdk_aac_jll
   1743.6 ms  ✓ ColorVectorSpace
    547.3 ms  ✓ libdrm_jll
    533.6 ms  ✓ LAME_jll
    497.2 ms  ✓ fzf_jll
    399.0 ms  ✓ mtdev_jll
    470.4 ms  ✓ Ogg_jll
    492.4 ms  ✓ x265_jll
    497.7 ms  ✓ x264_jll
    522.4 ms  ✓ libaom_jll
    529.1 ms  ✓ Expat_jll
    465.0 ms  

In [2]:

# Inclure tes fichiers locaux
include("../DataAssim.jl/src/lorenz95.jl")
include("../DataAssim.jl/src/operators.jl")

   Resolving package versions...
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Project.toml`
  No Changes to `~/Desktop/DataAssim.jl/Data_Assimilation/Manifest.toml`


build_model (generic function with 1 method)

In [3]:
using NLPModels, ADNLPModels
using JSOSolvers

function nonlinear_funcval(x, y, obs, xb, B, R)
    eo = y .- gop(obs, x)
    eb = x .- xb
    return (1/2) * eb'* invdot(B, eb) + (1/2) * eo'*invdot(R, eo)
end

nonlinear_funcval (generic function with 1 method)

In [4]:
n = 1000
nt = 8
dt = 0.025
F = 8.0
rng = MersenneTwister(1234)

sigmaR =  0.1
total_space_obs = 300
total_time_obs = 2   # m = total_space_obs*total_time_obs
sigmaB =  0.8

# Model
model = Lorenz95Model(F, dt)

# Observations

space_inds_obs = round.(Int, range(1, n; length=total_space_obs))
time_inds_obs = round.(Int, range(1, nt-1; length=total_time_obs))
m = total_space_obs*total_time_obs
obs = ObsOperator(sigmaR, space_inds_obs, n, time_inds_obs, nt, m, model)
R = RMatrix(sigmaR)

# Background
B = BMatrix(sigmaB, n)

# Spin-up
xt = 3.0 .* ones(n) .+ randn(rng, n)
xt = traj(model, xt, 5000)

# Background state
xb = xt .+ randn(rng, n) .* sigmaB

# Observations
y = generate_obs(obs, xt)

# Construct ADNLPModel
f(x) = nonlinear_funcval(x, y, obs, xb, B, R)

f (generic function with 1 method)

In [5]:
using DataFrames
function run_solver(nlp, subsolver; memory=nothing)

    if memory === nothing
        stats = trunk(nlp,
            max_time=10000.0,
            max_iter=500,
            verbose=0,
            subsolver=subsolver
        )
    else
        stats = trunk(nlp,
            max_time=10000.0,
            max_iter=500,
            verbose=0,
            subsolver=subsolver,
            subsolver_kwargs=(memory=memory,)
        )
    end

    row = (
        solver = string(subsolver) * (memory === nothing ? "" : "_m$(memory)"),
        status = stats.status,
        norm_sol = norm(stats.solution),
        objective = stats.objective,
        iter = stats.iter,
        obj_eval = nlp.counters.neval_obj,
        grad_eval = nlp.counters.neval_grad,
        hprod = nlp.counters.neval_hprod,
        time = stats.elapsed_time
    )

    reset!(nlp)

    return row
end

run_solver (generic function with 1 method)

In [6]:
x0 =  xb
nlp = ADNLPModel(f, x0)

ADNLPModel - Model with automatic differentiation backend ADModelBackend{
  ForwardDiffADGradient,
  ForwardDiffADHvprod,
  EmptyADbackend,
  EmptyADbackend,
  EmptyADbackend,
  SparseADHessian,
  EmptyADbackend,
}
  Problem name: Generic
   All variables: ████████████████████ 1000   All constraints: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            free: ████████████████████ 1000              free: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                lower: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                upper: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
         low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0              low/upp: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
           fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0                fixed: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
          infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0               infeas: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
            nnzh: ( 83.16% sparsity)   84300           linear: ⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅⋅ 0     
                                 

In [ ]:
results = DataFrame()

push!(results, run_solver(nlp, :cg))
push!(results, run_solver(nlp, :lbfgs; memory=100))
push!(results, run_solver(nlp, :diom; memory=100))
push!(results, run_solver(nlp, :lbfgs; memory=50))
push!(results, run_solver(nlp, :diom; memory=50))

Row,solver,status,norm_sol,objective,iter,obj_eval,grad_eval,hprod,time
,String,Symbol,Float64,Float64,Int64,Int64,Int64,Int64,Float64
1,cg,first_order,140.244,339.606,29,117,28,689,403.371
2,lbfgs_m100,first_order,140.256,339.457,34,132,31,522,311.677
3,diom_m100,first_order,140.256,339.457,34,132,31,522,328.685
4,lbfgs_m50,first_order,140.256,339.457,34,132,31,522,321.986
5,diom_m50,first_order,140.256,339.457,34,132,31,522,317.491


In [14]:
using PrettyTables
pretty_table(results)

┌────────────┬─────────────┬──────────┬───────────┬───────┬──────────┬──────────
│     solver │      status │ norm_sol │ objective │  iter │ obj_eval │ grad_ev ⋯
│     String │      Symbol │  Float64 │   Float64 │ Int64 │    Int64 │     Int ⋯
├────────────┼─────────────┼──────────┼───────────┼───────┼──────────┼──────────
│         cg │ first_order │  140.244 │   339.606 │    29 │      117 │         ⋯
│ lbfgs_m100 │ first_order │  140.256 │   339.457 │    34 │      132 │         ⋯
│  diom_m100 │ first_order │  140.256 │   339.457 │    34 │      132 │         ⋯
│  lbfgs_m50 │ first_order │  140.256 │   339.457 │    34 │      132 │         ⋯
│   diom_m50 │ first_order │  140.256 │   339.457 │    34 │      132 │         ⋯
└────────────┴─────────────┴──────────┴───────────┴───────┴──────────┴──────────
                                                               3 columns omitted
